# Saddle-Mediated Transport — Sigmoid Projection Along the Transit Axis

**Hypothesis.** Training dynamics pass through a saddle. The saddle axis is rotated from the principal components of the parameter trajectory. One end of the axis is the *wiggle-point* (the deflection right after first descent, which coincides with a peak in MLP dimensionality). The other end is the final checkpoint.

If that picture is right, projecting the full-parameter trajectory onto the unit vector `(terminal − wiggle)` should yield a sigmoid over epochs. Its derivative should peak at the saddle crossing.

**Reference variant.** `p109/s485/ds598` — reference healthy model with dense checkpointing (every 100 epochs through epoch 10000). `first_descent_end=1200`, `second_descent_onset=3584`, `grokking=5243`.

**Parameter space.** MLP only — `W_in` + `W_out` flattened and concatenated (the `COMPONENT_GROUPS['mlp']` bundle). Attention and embedding trajectories don't show the deflection, so scoping to MLP keeps the signal uncontaminated.

In [ ]:
import json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from miscope import load_family
from miscope.analysis.artifact_loader import ArtifactLoader
from miscope.analysis.library.weights import COMPONENT_GROUPS
from miscope.visualization.renderers.dimensionality_dynamics import (
    _compute_rolling_trajectory_metrics,
)
from notebooks.sketch_per_group_kinks import run_sketch

FAMILY_NAME = 'modulo_addition_1layer'
family = load_family(FAMILY_NAME)

MANUAL = {
    'p109/s485/ds598 - fast, clean': {'prime': 109, 'seed': 485, 'data_seed': 598, 'start': [1700], 'end': [20000]},
    'p113/s999/ds598 - canon': {'prime': 113, 'seed': 999, 'data_seed': 598, 'start': [2000], 'end': [20000]},
    'p59/s485/ds598 - super-critical': {'prime': 59, 'seed': 485, 'data_seed': 598, 'start': [1700], 'end': [29000]},
    'p101/s485/ds999 - rebounder': {'prime': 101, 'seed': 485, 'data_seed': 999, 'start': [1800], 'end': [18500]},
    'p101/s999/ds598 - long, bumpy descent': {'prime': 101, 'seed': 999, 'data_seed': 598, 'start': [2100], 'end': [29200]},
}
    

In [44]:
# Helper methods
def load_mlp_param_trajectory(variant):
    """Load W_in ⊕ W_out flattened per epoch — full MLP parameter space."""
    loader = ArtifactLoader(str(variant.variant_dir / 'artifacts'))
    epochs = sorted(loader.get_epochs('parameter_snapshot'))
    mlp_keys = COMPONENT_GROUPS['mlp']  # ['W_in', 'W_out']
    rows = []
    for e in epochs:
        snap = loader.load_epoch('parameter_snapshot', e)
        rows.append(np.concatenate([snap[k].flatten() for k in mlp_keys]))
    X = np.stack(rows).astype(np.float64)
    return np.array(epochs), X

def compute_transit_projection(X, epochs, wiggle_epoch, terminal_epoch):
    """Project full trajectory onto unit transit vector; normalize so wiggle→0, terminal→1."""
    i_w = int(np.where(epochs == wiggle_epoch)[0][0])
    i_T = int(np.where(epochs == terminal_epoch)[0][0])
    delta = X[i_T] - X[i_w]
    L = np.linalg.norm(delta)
    v_hat = delta / L
    # s normalized so s=0 at wiggle, s=1 at terminal
    s = (X - X[i_w]) @ v_hat / L
    return s, L, i_w, i_T


def compute_velocity(s, epochs):
    """ds/dt using central differences scaled by per-epoch gap."""
    s = np.asarray(s, dtype=np.float64)
    ep = np.asarray(epochs, dtype=np.float64)
    ds_dt = np.zeros_like(s)
    ds_dt[1:-1] = (s[2:] - s[:-2]) / (ep[2:] - ep[:-2])
    ds_dt[0]    = (s[1] - s[0]) / (ep[1] - ep[0])
    ds_dt[-1]   = (s[-1] - s[-2]) / (ep[-1] - ep[-2])
    return ds_dt

def plot_sigmoid_and_velocity(manual_model, epochs, X, s, v, export=False):
    # Sigmoid + velocity plot
    label = manual_model[0]
    
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                        subplot_titles=['s(t)  —  scalar projection along transit axis  (0=wiggle, 1=terminal)',
                                        'ds/dt  —  projection velocity'])

    fig.add_trace(go.Scatter(x=epochs, y=s, mode='lines+markers', name='s(t)',
                            line=dict(color='#1f77b4', width=2),
                            marker=dict(size=4)), row=1, col=1)
    fig.add_hline(y=0.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
    fig.add_hline(y=1.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
    fig.add_hline(y=0.5, line=dict(color='rgba(0,0,0,0.15)', dash='dot', width=1), row=1, col=1)

    fig.add_trace(go.Scatter(x=epochs, y=v, mode='lines+markers', name='ds/dt',
                            line=dict(color='#d62728', width=2),
                            marker=dict(size=4)), row=2, col=1)

    for row in (1, 2):
        fig.add_vline(x=manual_model[1]['fd_end'],    line=dict(color='orange', dash='dot', width=1), row=row, col=1)
        fig.add_vline(x=manual_model[1]['sd_onset'],  line=dict(color='teal',   dash='dot', width=1), row=row, col=1)
        fig.add_vline(x=manual_model[1]['grok'],      line=dict(color='black',  dash='dash', width=1), row=row, col=1)
        fig.add_vline(x=manual_model[1]["start"][0],  line=dict(color='red',    dash='solid', width=1.5), row=row, col=1)
        fig.add_vline(x=manual_model[1]['saddle_epoch'],        line=dict(color='green',  dash='solid', width=1.5), row=row, col=1)

    fig.update_yaxes(title_text='s (normalized)', row=1, col=1)
    fig.update_yaxes(title_text='ds/dt',         row=2, col=1)
    fig.update_xaxes(title_text='epoch', row=2, col=1)
    fig.update_layout(
        title=(f'{label} — transit projection  (wiggle ep={manual_model[1]["start"][0]}, terminal ep={manual_model[1]["end"][0]})<br>'
            '<sup>orange=fd_end · teal=sd_onset · black=grok · red=wiggle · green=ds/dt peak</sup>'),
        template='plotly_white', height=700, width=1000, showlegend=False,
    )
    if export:
        file_label = f"p{manual_model[1]["prime"]}_s{manual_model[1]["seed"]}_ds{manual_model[1]["data_seed"]}"
        fig.write_image(file=f"exports/saddle_transport/transit_projection_manual_{file_label}.png", format="png")
    fig.show()

## 1. Transit vector and scalar projection - 1 plot per model

For a chosen wiggle epoch `e_w` and terminal epoch `e_T`:

```
v = (X[e_T] − X[e_w]) / ‖X[e_T] − X[e_w]‖
s(t) = (X[t] − X[e_w]) · v     ≡     scalar distance along v from wiggle
```

Normalizing by `‖X[e_T] − X[e_w]‖` so `s=0` at wiggle and `s=1` at terminal. If the saddle picture is right, `s(t)` is sigmoidal with inflection at the crossing.

In [46]:
# iterate models
for model in MANUAL.items():
    #WIGGLE_EPOCH = model[1]['start'][0]           # primary candidate — rolling PR₃ peak
    #TERMINAL_EPOCH = model[1]['end'][0]

    variant = family.get_variant(prime=model[1]['prime'], seed=model[1]['seed'], data_seed=model[1]['data_seed'])

    with open(variant.variant_dir / 'variant_summary.json') as f:
        summary = json.load(f)
    model[1]['fd_end'] = summary['first_descent_window']['end_epoch']
    model[1]['sd_onset'] = summary.get('second_descent_onset_epoch')
    model[1]['grok'] = summary['second_descent_window']['end_epoch']
    model[1]['effdim_crossover'] = summary.get('effective_dimensionality_cross_over_epoch')

    epochs, X = load_mlp_param_trajectory(variant)
    # model[1]["epochs"] = epochs
    # model[1]["X"] = X
    s, L, i_w, i_T = compute_transit_projection(X, epochs, model[1]['start'][0], model[1]['end'][0])
    # model[1]["s"] = s
    model[1]["L"] = L
    model[1]["i_w"] = i_w
    model[1]["i_T"] = i_T
    v = compute_velocity(s, epochs)
    saddle_idx = int(np.argmax(v))
    saddle_epoch = int(epochs[saddle_idx])

    # model[1]["v"] = v
    model[1]["saddle_epoch"] = saddle_epoch

    print(f'model: p{model[1]["prime"]}/s{model[1]["seed"]}/ds{model[1]["data_seed"]}')
    print(f'wiggle epoch:     {model[1]['start'][0]}')
    print(f'terminal epoch:   {model[1]['end'][0]}')
    print(f'transit length:   {L:.3f}  (‖W_T − W_w‖ in MLP space)')
    print(f'velocity-peak ep: {saddle_epoch}  (ds/dt max = {v[saddle_idx]:.2e})')
    print(f's at wiggle = {s[i_w]:.3f}, s at terminal = {s[i_T]:.3f}  (should be 0.0 and 1.0)')

    plot_sigmoid_and_velocity(manual_model=model, epochs=epochs, X=X, s=s, v=v, export=True)

model: p109/s485/ds598
wiggle epoch:     1700
terminal epoch:   20000
transit length:   30.616  (‖W_T − W_w‖ in MLP space)
velocity-peak ep: 1500  (ds/dt max = 3.13e-04)
s at wiggle = 0.000, s at terminal = 1.000  (should be 0.0 and 1.0)


model: p113/s999/ds598
wiggle epoch:     2000
terminal epoch:   20000
transit length:   33.535  (‖W_T − W_w‖ in MLP space)
velocity-peak ep: 1500  (ds/dt max = 2.34e-04)
s at wiggle = 0.000, s at terminal = 1.000  (should be 0.0 and 1.0)


model: p59/s485/ds598
wiggle epoch:     1700
terminal epoch:   29000
transit length:   26.849  (‖W_T − W_w‖ in MLP space)
velocity-peak ep: 1400  (ds/dt max = 4.23e-04)
s at wiggle = 0.000, s at terminal = 1.000  (should be 0.0 and 1.0)


model: p101/s485/ds999
wiggle epoch:     1800
terminal epoch:   18500
transit length:   31.261  (‖W_T − W_w‖ in MLP space)
velocity-peak ep: 1500  (ds/dt max = 2.75e-04)
s at wiggle = 0.000, s at terminal = 1.000  (should be 0.0 and 1.0)


model: p101/s999/ds598
wiggle epoch:     2100
terminal epoch:   29200
transit length:   31.690  (‖W_T − W_w‖ in MLP space)
velocity-peak ep: 1400  (ds/dt max = 2.53e-04)
s at wiggle = 0.000, s at terminal = 1.000  (should be 0.0 and 1.0)


In [ ]:
run_sketch(prime=113, seed=999, data_seed=598, variant_label="canon")
run_sketch(prime=101, seed=999, data_seed=999, variant_label="bumpy descent with slight rebound")
